In [20]:
authors = "El Majdoub F, Sayed D, Surges G, Rosenberg WS"

if authors and len(authors) > 0:
        if isinstance(authors, str):
            first_author = authors.split(',')[0].strip() # format of Last FI, Last FI
        else:
            first_author = authors[0].strip()
        # Extract last name (assume format "LastName FirstInitial" or "LastName, FirstName")
        print(first_author)
        author_parts = first_author.replace(',', '').split()
        print(author_parts)
        print(len(author_parts))
        if author_parts:
            if len(author_parts) > 2:
                author_last_name = ' '.join(author_parts[:-1])
            else:
                author_last_name = author_parts[0]
author_last_name

El Majdoub F
['El', 'Majdoub', 'F']
3


'El Majdoub'

In [5]:
from time import sleep
import requests
from pprint import pprint
import matplotlib.pyplot as plt
from datetime import datetime

ESUMMARY_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
params = {
    'db': 'pubmed',
    'id': '27933103',
    'retmode': 'json'
}

try:
    response = requests.get(ESUMMARY_URL, params=params)
    # Raise an error for bad responses
    response.raise_for_status()  
    data = response.json()
    pprint(data, depth=3)
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")
    data = None


# Check if we got data back from the API
if data:
    authors = data["result"]["27933103"]["authors"]
    # Loop through the authors and print their names
    for author in authors:
        print(f"{author['name']}")
    for param in data["result"]["27933103"]:
        print(f"{param}: {data['result']['27933103'][param]}")
    title = data["result"]["27933103"]["title"]
    print(title)

{'header': {'type': 'esummary', 'version': '0.3'},
 'result': {'27933103': {'articleids': [...],
                         'attributes': [...],
                         'authors': [...],
                         'availablefromurl': '',
                         'bookname': '',
                         'booktitle': '',
                         'chapter': '',
                         'doccontriblist': [],
                         'docdate': '',
                         'doctype': 'citation',
                         'edition': '',
                         'elocationid': '',
                         'epubdate': '2016 Nov 23',
                         'essn': '1758-2946',
                         'fulljournalname': 'Journal of cheminformatics',
                         'history': [...],
                         'issn': '1758-2946',
                         'issue': '',
                         'lang': [...],
                         'lastauthor': 'Bara JE',
                         'location

In [25]:

from pydantic import BaseModel, Field
from typing import Optional, List

import re
import requests
import time
from typing import Optional, Dict, List, Tuple
from urllib.parse import quote, urlparse
from dataclasses import dataclass

# from utils.logging_config import logger

class EnrichedPaperMetadata(BaseModel):
    """Enhanced metadata including PubMed lookup results."""
    # Original extracted features
    title: str
    authors: str

    # PubMed lookup results
    pubmed_lookup_success: bool
    pubmed_id: Optional[str] = None
    pubmed_url: Optional[str] = None
    pubmed_validated: bool = False
    lookup_error: Optional[str] = None
    pubmed_journal: Optional[str] = None
    pubmed_year: Optional[str] = None
    pubmed_doi: Optional[str] = None
    
    
class PubMedAPI:
    """Interface to PubMed E-utilities API."""

    BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
    SEARCH_URL = f"{BASE_URL}/esearch.fcgi" # use for PMID lookup
    FETCH_URL = f"{BASE_URL}/efetch.fcgi"
    SUMMARY_URL = f"{BASE_URL}/esummary.fcgi" # use for meta data retrieval

    def _make_request(self, url: str, params: dict) -> requests.Response: 

        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            return response
        except requests.RequestException as e:
            # logger.error(f"PubMed API request failed: {str(e)}")
            raise

    def search(self, query: str, max_results: int = 10) -> List[str]:
        """
        Search PubMed for papers matching the query.

        Args:
            query: Search query string
            max_results: Maximum number of results to return

        Returns:
            List of PMIDs (PubMed IDs)
        """
        params = {
            'db': 'pubmed',
            'term': query,
            'retmode': 'json',
            'retmax': max_results
        }

        try:
            response = self._make_request(self.SEARCH_URL, params)
            data = response.json()

            pmids = data.get('esearchresult', {}).get('idlist', [])
            # logger.debug(f"PubMed search found {len(pmids)} results for query: {query[:100]}")
            return pmids

        except Exception as e:
            # logger.error(f"PubMed search failed: {str(e)}")
            return []

    def find_pmid_by_metadata(self, authors: str, title: str, year: Optional[int] = None,
                              doi: Optional[str] = None, max_results: int = 5) -> Optional[str]:
        """
        Find PMID by searching with author names, title, year, and DOI.

        Args:
            authors: List of author names (e.g., ["Smith J", "Doe A", "Johnson B"])
            title: Paper title
            year: Publication year (optional)
            doi: DOI link/string (optional, prioritized if provided)
            max_results: Maximum number of results to return from search

        Returns:
            PMID string if found, None otherwise
        """
        query_parts = []

        # If DOI is provided, prioritize it as it's the most reliable identifier
        if doi:
            doi_clean = doi.strip() #remove whitespace
            if 'doi.org/' in doi_clean:
                doi_clean = doi_clean.split('doi.org/')[-1] #remove doi.org if present
            query_parts.append(f'{doi_clean}[DOI]') # [DOI] tag for search with doi tag
            # logger.info(f"Searching with DOI: {doi_clean}")

        # Add title to query
        if title:
            title_clean = title.strip() #remove whitespace
            query_parts.append(f'{title_clean}[Title]') # Tag

        # Add first author's last name if available
        if authors and len(authors) > 0:
            if isinstance(authors, str):
                first_author = authors.split(',')[0].strip() # format of Last FI, Last FI
            else:
                first_author = authors[0].strip() # list handling
            
            author_parts = first_author.replace(',', '').split() # remove any commas and split by space
            if author_parts:
                if len(author_parts) > 2:
                    author_last_name = ' '.join(author_parts[:-1]) # for e.g. El Madjoub 
                else:
                    author_last_name = author_parts[0]
                query_parts.append(f'{author_last_name}[Author]') # author tag

        # Add year if provided
        if year:
            query_parts.append(f'{year}[pdat]') # pub date tag

        # final query
        query = ' AND '.join(query_parts)
        # logger.info(f"Searching PubMed with query: {query}")

        # Make request to SEARCH_URL
        params = {
            'db': 'pubmed',
            'term': query,
            'retmode': 'json',
            'retmax': max_results
        }

        # Make request with params
        try:
            response = self._make_request(self.SEARCH_URL, params)
            data = response.json()

            pmids = data.get('esearchresult', {}).get('idlist', []) #dict access with no KeyError

            if pmids:
                pmid = pmids[0]  # Return the first result, most relevant
                # logger.info(f"Found PMID: {pmid}")
                return pmid
            
            else:
                # logger.warning(f"No PMID found for query: {query}")
                return None

        except Exception as e:
            # logger.error(f"Failed to find PMID: {str(e)}")
            return None
    
    
    def lookup_and_validate(title: str, authors: str, year: Optional[int] = None,
                       email: Optional[str] = None) -> Dict[str, any]:
        """
        Complete pipeline: search PubMed, find paper, validate link.

        Args:
            title: Paper title
            authors: Author string
            year: Publication year (optional)
            email: Email for NCBI API (optional)

        Returns:
            Dictionary with results:
            {
                'found': bool,
                'pmid': str or None,
                'pubmed_url': str or None,
                'validated': bool,
                'paper': PubMedPaper or None,
                'error': str or None
            }
        """
        result = {
            'found': False,
            'pmid': None,
            'pubmed_url': None,
            'validated': False,
            'paper': None,
            'error': None
        }

        try:
            # Initialize API client
            api = PubMedAPI(email=email)

            # Search for paper
            logger.info(f"Looking up paper: {title[:60]}...")
            paper = api.search_by_title_author(title, authors, year)

            if not paper:
                result['error'] = "Paper not found in PubMed"
                logger.warning(result['error'])
                return result

            result['found'] = True
            result['pmid'] = paper.pmid
            result['pubmed_url'] = paper.pubmed_url
            result['paper'] = paper

            # Validate the link
            is_valid, pmid = validate_pubmed_link(paper.pubmed_url)
            result['validated'] = is_valid

            if is_valid:
                logger.info(f"Successfully found and validated PubMed paper: {paper.pubmed_url}")
            else:
                result['error'] = "PubMed link found but validation failed"
                logger.warning(result['error'])

            return result

        except Exception as e:
            error_msg = f"PubMed lookup failed: {str(e)}"
            result['error'] = error_msg
            logger.error(error_msg)
            return result

In [29]:
PubMedAPI().find_pmid_by_metadata(authors=authors, title="A multicenter real-world review of 10 kHz SCS outcomes")

def get_metadata_by_pmid(self, pmid: str) -> Optional[Dict[str, str]]:
    """
    Retrieve paper metadata from PubMed using PMID.

    Args:
        pmid: PubMed ID string
    """
    
    params = {
        'db': 'pubmed',
        'id': pmid,
        'retmode' : 'json'
    }
    try:
        response = requests.get(ESUMMARY_URL, params=params)
        response.raise_for_status()
        data = response.json()
        data_short = data["result"][pmid]
        title = data_short["title"]
        authors = ', '.join(data_short["authors"][i]["name"] for i in range(len(data_short["authors"])))
        print(f'Title: {title}')
        print(f'Authors: {authors}')
    except Exception as e:
        logger.error(f"Failed to fetch metadata for PMID {pmid}: {str(e)}")
        return None

get_metadata_by_pmid(PubMedAPI(), pmid="30911573")

Title: A multicenter real-world review of 10 kHz SCS outcomes for treatment of chronic trunk and/or limb pain.
Authors: Stauss T, El Majdoub F, Sayed D, Surges G, Rosenberg WS, Kapural L, Bundschu R, Lalkhen A, Patel N, Gliner B, Subbaroyan J, Rotte A, Edgar DR, Bettag M, Maarouf M
